# Destiny Explained Studio — Athéon et Caveau de verre — Cinematic 0.9.1

Extension visuelle originale. Choisir **Exécution → Modifier le type d’exécution → GPU T4** si disponible, puis exécuter les cellules dans l’ordre. Fonctionne aussi sur CPU, plus lentement. Aucun épisode complet, aucun média du jeu téléchargé. Les sorties restent dans `/content`, téléchargeables à la fin ; aucune connexion Drive nécessaire.


In [ ]:
# @title 1. Installer l’extension et Blender officiel (à exécuter une fois)
import hashlib, pathlib, subprocess, urllib.request
REPO = pathlib.Path('/content/Destiny-Explained-Studio')
BRANCH = 'feat/cinematic-vex-overhaul-0.9.1'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/Alexanst54/Destiny-Explained-Studio.git',str(REPO)],check=True)
else:
    actual = subprocess.check_output(['git','-C',str(REPO),'branch','--show-current'],text=True).strip()
    assert actual == BRANCH, 'Le dossier existe sur une autre branche ; utilisez un runtime neuf.'
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','libxi6','libxrender1','libxkbcommon0','libsm6','libice6','libgl1','libegl1','ffmpeg'],check=True)
archive = pathlib.Path('/content/blender-4.3.2-linux-x64.tar.xz')
base = 'https://download.blender.org/release/Blender4.3/'
BLENDER = pathlib.Path('/content/blender-4.3.2-linux-x64/blender')
if not BLENDER.exists():
    urllib.request.urlretrieve(base+archive.name,str(archive))
    checksums = urllib.request.urlopen(base+'blender-4.3.2.sha256').read().decode()
    checksum = hashlib.file_digest(archive.open('rb'),'sha256').hexdigest()
    assert any(checksum in line and archive.name in line for line in checksums.splitlines()), 'SHA-256 incorrect'
    subprocess.run(['tar','-xf',str(archive),'-C','/content'],check=True)
subprocess.run([str(BLENDER),'--version'],check=True)
try: subprocess.run(['nvidia-smi'],check=False)
except FileNotFoundError: print('Aucun pilote NVIDIA détecté ; fallback CPU disponible.')


In [ ]:
# @title 2. Choisir l’asset Atlas et rendre (nouveau dossier à chaque exécution)
selection = 'boss-atheon' # @param ['venus','vault-of-glass','arena-templars-well','arena-glass-throne','boss-templar','boss-atheon']
lod = 0 # @param [0,1,2] {type:'raw'}
# LOD 0 conserve toute la géométrie même en basse résolution.
quality = 'balanced' # @param ['draft','balanced','cinematic']
camera = 'reveal' # @param ['wide','reveal','orbit','low-angle','close-up','front','profile','ground-reveal','mechanism','temporal']
animation = 'idle' # @param ['idle','walk','combat','temporal','rotate','move','attack','reveal']
seed = 42 # @param {type:'integer'}
frames = 1 # @param [1,24,48,60] {type:'raw'}
width = 960 # @param [480,640,960,1280,1920] {type:'raw'}
height = 540 # @param [270,360,540,720,1080] {type:'raw'}
start_frame = 0 # @param {type:'integer'}
# Athéon: idle/walk/combat/temporal. Templier: idle/rotate/move/attack.
# Décor seul: idle. Une mauvaise association est rejetée avant Blender.
import datetime
OUT = pathlib.Path('/content/venus-vault-renders') / datetime.datetime.now().strftime('%Y%m%d-%H%M%S-%f')
command = ['python',str(REPO/'remotion/blender/render_venus_vault.py'),selection,str(OUT),
           '--blender',str(BLENDER),'--lod',str(lod),'--quality',quality,'--camera',camera,'--animation',animation,
           '--seed',str(seed),'--frames',str(frames),'--fps','24','--width',str(width),'--height',str(height),
           '--start-frame',str(start_frame),'--threads','2','--timeout','7200']
if frames > 1: command.append('--video')
subprocess.run(command,cwd=REPO,check=True)


In [ ]:
# @title 3. Inspecter les vrais rendus, le rapport et télécharger
import json, shutil
from IPython.display import display, Image, Video
from google.colab import files
display(Image(filename=str(OUT/'frame-0000.png')))
if (OUT/'sequence.mp4').exists(): display(Video(str(OUT/'sequence.mp4'),embed=True))
report = json.loads((OUT/'render-report.json').read_text())
print(json.dumps({k:report[k] for k in ['status','request','configuration','secondsInsideBlender','peakProcessRssKiB']},indent=2))
print('Le rapport indique le backend réellement utilisé ; aucun gain GPU n’est présumé.')
archive_out = shutil.make_archive(str(OUT), 'zip', OUT)
files.download(archive_out)


## Limites

Reconstruction voxel originale, pas copie de la carte ni des modèles officiels. Pas de narration ni de sous-titres dans le clip de validation ; Kokoro/Remotion historiques restent inchangés. Le rendu conserve la scène durant cette courte séquence, mais ne reprend pas un processus interrompu : relancer dans un dossier neuf. T4/Colab/Android demandent une validation sur ce matériel ; les rendus livrés dans le dépôt ont été mesurés sur CPU. Pour un premier essai, conserver une seule image ; ne pas lancer un épisode complet.
